# 04. Atribuição simples — mãe + endereço

Lê as predictions do
[`02d_aplicar_splink_mae_endereco.ipynb`](02d_aplicar_splink_mae_endereco.ipynb).

**Sem escada.** Regras:

1. Piso de nota (`PISO`, default 0,50).
2. Melhor CPF por Censo: maior `match_probability`; empate → mãe
   preenchida nos dois lados → logradouro igual → CEP igual.
3. Um CPF leva no máximo 3 Censos; se no topo tiver 4+, nenhum fica.

Exporta `splink_atribuicao_mae_endereco.parquet`.


In [ ]:
import sys
from pathlib import Path

PROB_DIR = Path.cwd()
if PROB_DIR.name == 'notebooks':
    PROB_DIR = PROB_DIR.parent
if str(PROB_DIR) not in sys.path:
    sys.path.insert(0, str(PROB_DIR))

from config import (
    MAX_CENSOS_POR_CPF,
    SPLINK_ATRIBUICAO_MAE_ENDERECO,
    SPLINK_INPUT_VIEW,
    SPLINK_PREDICTIONS_MAE_ENDERECO,
    TABELA_CENSO_LIMPA_APLICACAO,
    TABELA_CPF_LIMPA_APLICACAO,
    drop_splink_temp_tables,
    export_parquet,
    get_connection,
    materialize_splink_input,
    print_paths,
    require_input,
    require_tables,
)

PISO = 0.50
TETO = MAX_CENSOS_POR_CPF

print_paths()
require_input(
    SPLINK_PREDICTIONS_MAE_ENDERECO,
    label='SPLINK_PREDICTIONS_MAE_ENDERECO (rode o 02d antes)',
)
con = get_connection()
drop_splink_temp_tables(con)
require_tables(
    con,
    [TABELA_CENSO_LIMPA_APLICACAO, TABELA_CPF_LIMPA_APLICACAO],
    notebook_origem='00b',
)
materialize_splink_input(
    con,
    censo_table=TABELA_CENSO_LIMPA_APLICACAO,
    cpf_table=TABELA_CPF_LIMPA_APLICACAO,
)
print('Piso:', PISO, '| teto de Censos por CPF:', TETO)
print('Predictions:', SPLINK_PREDICTIONS_MAE_ENDERECO)


In [ ]:
_tipo = con.execute('''
SELECT table_type
FROM information_schema.tables
WHERE table_schema = 'main' AND table_name = 'splink_predictions'
''').fetchone()
if _tipo:
    _kind = 'VIEW' if _tipo[0].upper() == 'VIEW' else 'TABLE'
    con.execute(f'DROP {_kind} IF EXISTS splink_predictions')
con.execute(f'''
CREATE OR REPLACE VIEW splink_predictions AS
SELECT
    CASE
        WHEN unique_id_l LIKE 'censo_%' THEN unique_id_l
        ELSE unique_id_r
    END AS unique_id_censo,
    CASE
        WHEN unique_id_l LIKE 'censo_%' THEN unique_id_r
        ELSE unique_id_l
    END AS unique_id_cpf,
    match_probability
FROM read_parquet('{SPLINK_PREDICTIONS_MAE_ENDERECO}')
''')
n_pred = con.execute('SELECT COUNT(*) FROM splink_predictions').fetchone()[0]
print('Pares nas predictions:', f'{n_pred:,}')


## Melhor nota + teto 3

Por Censo fica o CPF de maior nota. Empate: mãe preenchida nos dois lados,
depois logradouro igual, depois CEP igual. Depois, só entram CPFs com no
máximo 3 Censos nesse topo.


In [ ]:
con.execute(f'''
CREATE OR REPLACE TABLE melhor_por_censo AS
SELECT
    p.unique_id_censo,
    p.unique_id_cpf,
    p.match_probability
FROM splink_predictions p
JOIN {SPLINK_INPUT_VIEW} ca ON ca.unique_id = p.unique_id_censo
JOIN {SPLINK_INPUT_VIEW} pb ON pb.unique_id = p.unique_id_cpf
WHERE p.match_probability >= {PISO}
QUALIFY ROW_NUMBER() OVER (
    PARTITION BY p.unique_id_censo
    ORDER BY
        p.match_probability DESC,
        CAST(
            ca.nome_mae_phon IS NOT NULL AND pb.nome_mae_phon IS NOT NULL
            AS INTEGER
        ) DESC,
        CAST(
            ca.logradouro_norm IS NOT NULL AND pb.logradouro_norm IS NOT NULL
            AND ca.logradouro_norm = pb.logradouro_norm
            AS INTEGER
        ) DESC,
        CAST(
            ca.cep IS NOT NULL AND pb.cep IS NOT NULL AND ca.cep = pb.cep
            AS INTEGER
        ) DESC,
        p.unique_id_cpf
) = 1
''')

con.execute(f'''
CREATE OR REPLACE TABLE atribuicao AS
SELECT m.unique_id_censo, m.unique_id_cpf, m.match_probability
FROM melhor_por_censo m
JOIN (
    SELECT unique_id_cpf
    FROM melhor_por_censo
    GROUP BY 1
    HAVING COUNT(*) <= {TETO}
) c ON c.unique_id_cpf = m.unique_id_cpf
''')

n_melhor = con.execute('SELECT COUNT(*) FROM melhor_por_censo').fetchone()[0]
n_ok = con.execute('SELECT COUNT(*) FROM atribuicao').fetchone()[0]
n_app = con.execute(f'''
SELECT COUNT(*) FROM {SPLINK_INPUT_VIEW} WHERE origem = 'censo'
''').fetchone()[0]
print('Melhor par (após piso):', f'{n_melhor:,}')
print('Atribuídos (teto {TETO}):', f'{n_ok:,}')
print('Censo aplicação:', f'{n_app:,}', f'| cobertura {100 * n_ok / n_app:.1f}%')


In [ ]:
from IPython.display import display

display(con.execute('''
SELECT
    CASE
        WHEN match_probability >= 0.99 THEN 'p_ge_099'
        WHEN match_probability >= 0.90 THEN 'p_090_099'
        WHEN match_probability >= 0.75 THEN 'p_075_090'
        WHEN match_probability >= 0.50 THEN 'p_050_075'
        ELSE 'p_lt_050'
    END AS faixa,
    COUNT(*) AS n
FROM atribuicao
GROUP BY 1
ORDER BY 1
''').df())


In [ ]:
export_parquet(con, 'atribuicao', path=SPLINK_ATRIBUICAO_MAE_ENDERECO)
print('Exportado:', SPLINK_ATRIBUICAO_MAE_ENDERECO)
con.close()
